# Twitter Scraper 

### Luiz Verheyen 

In [143]:
# imports

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import re
import time
import pandas as pd
from datetime import datetime
import os
from dotenv import load_dotenv

In [144]:
# zorgen dat ik de env credentials kan gebruiken
load_dotenv() 

True

## Credentials needed for login
- email
- username
- password

In [145]:
my_twitter_email = os.getenv("my_twitter_email")
my_twitter_username = os.getenv("my_twitter_username")
my_twitter_password = os.getenv("my_twitter_password")

### options for the scraping configuration

In [146]:
options = uc.ChromeOptions() #initializing
options.add_argument("--start-maximized") # start screen volledig
options.add_argument("--disable-notifications") # disable alle notificaties die zouden binnenkomen

In [147]:
driver = uc.Chrome(version_main=143, options=options)
time.sleep(5)

In [148]:
# Ga naar loginpagina
driver.get("https://twitter.com/login")
time.sleep(5)  # wacht tot de pagina volledig geladen is

In [149]:
try:
    email_input = driver.find_element(By.NAME, "text")
    email_input.send_keys(my_twitter_email)
    time.sleep(1)
    email_input.send_keys(Keys.ENTER)
    time.sleep(3)
except:
    print("Geen Username input gevonden of al ingevuld.")
    
# 2 factor authentication
try:
    username_input = driver.find_element(By.NAME, "text")
    username_input.send_keys(my_twitter_username)
    time.sleep(1)
    username_input.send_keys(Keys.ENTER)
    time.sleep(3)
except:
    print("no 2 factor authentication found or not needed.")
    
# password in tikken
try:
    password_input = driver.find_element(By.NAME, "password")
    password_input.send_keys(my_twitter_password)
    time.sleep(1)
    password_input.send_keys(Keys.ENTER)
    time.sleep(3)
except:
    print("Geen wachtwoord invoer vereist of al ingelogd.")

In [150]:
def twitter_handle(username : str):
    driver.get(f"https://twitter.com/{username}")
    time.sleep(7)  # wachten tot pagina laadt
    # zorg dat cookies worden ge accepteert indien nodig:
    try:
        cookie_button = driver.find_element(By.XPATH, '//button[contains(., "Accept all cookies")]')
        cookie_button.click()
        print("Cookies geaccepteerd.")
        time.sleep(2)
    except:
        print("Geen cookie-wall gevonden of al geaccepteerd.")

In [151]:
def human_scroll(driver, total_scroll=3000, step=300, pause=0.3):
    scrolled = 0
    while scrolled < total_scroll:
        driver.execute_script(f"window.scrollBy(0, {step});")
        scrolled += step
        time.sleep(pause)

In [152]:
def save_to_csv(tweets_data):
    df = pd.DataFrame(tweets_data, columns=["Date", "Username", "Content", "Replies", "Reposts", "Likes", "Bookmarks", "Views"])
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d %H:%M')
    df['Time'] = pd.to_datetime(df['Date']).dt.time
    df['Date'] = pd.to_datetime(df['Date']).dt.date
    print(f"{len(df)} tweets opgeslagen!")
    return df

In [153]:
usernames_we_wanna_scrape = ["elonmusk", "realDonaldTrump"]
tweets_data = []

In [ ]:
# Instellingen
target_date = datetime.today().strftime("%Y-%m-%d")  # Stop met scrapen bij tweets ouder dan 1 jan 2026
scroll_pause = 2  # seconden tussen scrolls
tweets_ids = set() 

for user in usernames_we_wanna_scrape:
    stop_scraping = False
    twitter_handle(username=user)
    print(f"Start met scrapen tot aan {target_date} van {user}...")
    while stop_scraping == False:
        if stop_scraping:
            break
            
        # Zoek alle zichtbare tweets
        articles = driver.find_elements(By.XPATH, '//article[@role="article" and @data-testid="tweet"]')
        
        for article in articles:
            try:
                # 1. Datum ophalen
                time_elem = article.find_element(By.XPATH, './/time')
                date_str = time_elem.get_attribute("datetime")
                tweet_date = datetime.fromisoformat(date_str.replace("Z", "+00:00")).replace(tzinfo=None)

                # 2. Unieke ID check
                tweet_id = article.find_element(By.XPATH, './/time/..').get_attribute('href')
                tweet_username = tweet_id.split("/")[3]
                                
                if tweet_id in tweets_ids:
                    continue

                if tweet_username != user:
                    print(f"username: {tweet_username} is not {user}")
                    continue

                # 3. Check pinned
                try:
                    social_context = article.find_element(By.CSS_SELECTOR, "div[data-testid='socialContext']")
                    is_pinned = "Pinned" in social_context.text
                except:
                    is_pinned = False                    

                # 4. Beslissing opslaan
                # Opslaan alleen als pinned of tweet van vandaag
                if is_pinned and tweet_date.strftime("%Y-%m-%d") < target_date:
                    print(f"tweet: {tweet_date} is pinned and has older date")
                    continue
                elif tweet_date.strftime("%Y-%m-%d") < target_date:
                    stop_scraping = True
                    break

                # 5. Tekst en statistieken ophalen
                text = article.find_element(By.XPATH, './/div[@data-testid="tweetText"]').text
                stats_group = article.find_element(By.XPATH, './/div[@role="group"]')
                label = stats_group.get_attribute("aria-label")

                def parse_stat(pattern, text):
                    match = re.search(pattern, text.lower())
                    if match:
                        return int(re.sub(r'[^\d]', '', match.group(1)))
                    return 0

                replies = parse_stat(r'(\d[\d\.,]*)\s+replies', label)
                reposts = parse_stat(r'(\d[\d\.,]*)\s+reposts', label)
                likes = parse_stat(r'(\d[\d\.,]*)\s+likes', label)
                bookmarks = parse_stat(r'(\d[\d\.,]*)\s+bookmarks', label)
                views = parse_stat(r'(\d[\d\.,]*)\s+views', label)

                # Opslaan
                tweets_data.append([tweet_date, tweet_username, text, replies, reposts, likes, bookmarks, views])
                tweets_ids.add(tweet_id)
                print(f"Opgeslagen: {tweet_date.strftime('%Y-%m-%d %H:%M')} | @{tweet_username} | Likes: {likes}")

            except Exception as e:
                print(e)
                continue

        
        # Scroll naar beneden om nieuwe tweets te laden
        human_scroll(driver, total_scroll=2500, step=250, pause=0.4)
        time.sleep(scroll_pause)
        
driver.close()
df = save_to_csv(tweets_data=tweets_data)
print(f"Klaar! {len(tweets_data)} tweets verzameld.")

Cookies geaccepteerd.
Start met scrapen tot aan 2026-02-01 van elonmusk...
username: MarioNawfal is not elonmusk
username: AdamLowisz is not elonmusk
username: RyanSaavedra is not elonmusk
Opgeslagen: 2026-02-01 11:57 | @elonmusk | Likes: 43146
username: MarioNawfal is not elonmusk
username: AdamLowisz is not elonmusk
username: RyanSaavedra is not elonmusk
Opgeslagen: 2026-02-01 11:49 | @elonmusk | Likes: 30612
Opgeslagen: 2026-02-01 11:46 | @elonmusk | Likes: 79344
Opgeslagen: 2026-02-01 11:44 | @elonmusk | Likes: 18382
username: EvaFox is not elonmusk
username: EvaFox is not elonmusk
Opgeslagen: 2026-02-01 11:31 | @elonmusk | Likes: 32178
username: SBarrettBar is not elonmusk
username: Starlink is not elonmusk
Opgeslagen: 2026-02-01 08:26 | @elonmusk | Likes: 79341
username: SBarrettBar is not elonmusk
username: Starlink is not elonmusk
username: XFreeze is not elonmusk
Opgeslagen: 2026-02-01 07:57 | @elonmusk | Likes: 24199
Opgeslagen: 2026-02-01 07:46 | @elonmusk | Likes: 12548
use

In [155]:
df

,Date,Username,Content,Replies,Reposts,Likes,Bookmarks,Views,Time
0,2026-02-01,elonmusk,Legacy media lies relentlessly,7589,10080,43146,1854,4731004,11:57:00
1,2026-02-01,elonmusk,Electricity is proxy for industrial might.\n\n...,3179,7163,30612,1678,8789600,11:49:00
2,2026-02-01,elonmusk,Well said,1968,13209,79344,4904,5459471,11:46:00
3,2026-02-01,elonmusk,Yup,2128,3540,18382,367,2408856,11:44:00
4,2026-02-01,elonmusk,True,2755,6936,32178,1213,3564365,11:31:00
5,2026-02-01,elonmusk,Yes,5503,12951,79341,2462,6410572,08:26:00
6,2026-02-01,elonmusk,,3755,3161,24199,744,5784538,07:57:00
7,2026-02-01,elonmusk,True,1684,1712,12548,434,13174934,07:46:00
8,2026-02-01,elonmusk,I worked most of the weekend too. Still do. On...,4407,8346,72639,3277,9590482,06:55:00
9,2026-02-01,elonmusk,Accurate,2547,2714,24091,1193,23773241,06:20:00


In [160]:
dataFiles = [df, pd.read_csv("../../raw/elonmusk_tweets.csv")]

In [161]:
result = pd.concat(dataFiles, ignore_index=True)

In [162]:
result.set_index('Date', inplace=True)

In [163]:
result.to_csv("../../raw/elonmusk_tweets.csv", index=True)